In [6]:
import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
import random

# =========================
# CONFIGURATION
# =========================
initial_capital = 1_000_000
lot_size = 50
ATR_PERIOD = 10
ATR_MULTIPLIER = 2.5
BROKERAGE_PER_ORDER = 20
SLIPPAGE_PCT = 0.001
MARKET_CLOSE_TIME = '15:25:00'
VIX_LOW = 14
VIX_HIGH = 22
RISK_LOW = 0.15
RISK_MED = 0.10
RISK_HIGH = 0.05
CONFIRM_LOOKBACK = 10
MAX_LOSS = 18000
PARTIAL_BOOK_PCT = 0.2

BUY_PARTIAL_TP_PCT = 0.15
BUY_FINAL_TP_PCT_LOW = 0.20
BUY_FINAL_TP_PCT_HIGH = 0.40
BUY_ATR_PERIOD = 10
BUY_ATR_MULTIPLIER = 2.5
BUY_MAX_LOSS = 15000
BUY_BROKERAGE_PER_ORDER = 20
BUY_SLIPPAGE_PCT = 0.001
BUY_MARKET_CLOSE_TIME = '15:25:00'
BUY_VIX_LOW = 13
BUY_VIX_MED = 17
BUY_RISK_LOW = 0.15
BUY_RISK_MED = 0.10
BUY_RISK_HIGH = 0.05
BUY_IVR_THRESHOLD = 40

# =========================
# Class
# =========================
class UnifiedPortfolio:
    def __init__(self, capital):
        self.cash = capital
        self.open_positions = []
        self.trade_log = []
        self.brokerage = 0
        self.slippage = 0

    def can_execute(self, margin_required):
        return self.cash >= margin_required

    def allocate(self, margin):
        self.cash -= margin

    def release(self, margin):
        self.cash += margin

    def add_trade(self, trade):
        self.trade_log.append(trade)

    def add_costs(self, brokerage, slippage):
        self.brokerage += brokerage
        self.slippage += slippage

    def portfolio_value(self):
        return self.cash


# NEW Directory
# Updated File Paths for Out-of-Sample Data
VIX_PATH = r"D:\Bits Hyderabad_Campus Diaries\PS-1\AlgoBulls\Out_of_Sample_Data\Filtered_VIX_Jan_May_2025_1min.csv"
SIGNALS_PATH = r"D:\Bits Hyderabad_Campus Diaries\PS-1\AlgoBulls\Out_of_Sample_Data\Nifty_50_signals_15min_2025onwards_open_50_200.csv"
SPOT_PATH = r"D:\Bits Hyderabad_Campus Diaries\PS-1\AlgoBulls\Out_of_Sample_Data\nifty50_ohlcv_oi_2025_onwards.csv"
OPTIONS_PATH = r"D:\Bits Hyderabad_Campus Diaries\PS-1\AlgoBulls\Out_of_Sample_Data\Options_Sample_Jan_May_2025.csv"
# =========================
# DATA LOADING (Updated)
# =========================
def load_spot(start_date, end_date):
    df = pd.read_csv(SPOT_PATH, parse_dates=['TimeStamp'])
    df = df.set_index('TimeStamp')
    df.index = pd.to_datetime(df.index).tz_localize('Asia/Kolkata')
    # Adjust column names if needed ('Open', 'High', ...)
    df.columns = [col.lower() for col in df.columns]
    df = df.loc[start_date:end_date]
    return df

def load_signals_15min():
    signals = pd.read_csv(SIGNALS_PATH, parse_dates=['TimeStamp'])
    signals = signals.set_index('TimeStamp')
    if signals.index.tz is None:
        signals.index = pd.to_datetime(signals.index).tz_localize('Asia/Kolkata')
    else:
        signals.index = signals.index.tz_convert('Asia/Kolkata')
    if 'Signal' not in signals.columns:
        signals.columns = ['Signal']
    return signals

def load_options_sample():
    sample_path = OPTIONS_PATH
    options_sample = pd.read_csv(sample_path)
    options_sample['TimeStamp'] = pd.to_datetime(options_sample['TimeStamp'])
    if options_sample['TimeStamp'].dt.tz is None:
        options_sample['TimeStamp'] = options_sample['TimeStamp'].dt.tz_localize('Asia/Kolkata')
    else:
        options_sample['TimeStamp'] = options_sample['TimeStamp'].dt.tz_convert('Asia/Kolkata')
    options_sample['Expiry'] = pd.to_datetime(options_sample['Expiry'])
    return options_sample

def load_vix(start_date, end_date):
    vix = pd.read_csv(VIX_PATH, parse_dates=['TimeStamp'])
    vix = vix.set_index('TimeStamp')
    vix.index = pd.to_datetime(vix.index).tz_localize('Asia/Kolkata')
    vix = vix.loc[start_date:end_date]
    return vix

# =========================
# ADAPTIVE VIX FILTER (10th/90th percentile)
# =========================
def adaptive_vix_trend_filter(vix_series, current_time, window=252, low_pct=0.1, high_pct=0.9):
    vix_hist = vix_series.loc[:current_time].tail(window)
    if len(vix_hist) < window // 2:
        return False
    low_thr = np.percentile(vix_hist, low_pct*100)
    high_thr = np.percentile(vix_hist, high_pct*100)
    curr_vix = vix_series.loc[current_time]
    return (curr_vix > low_thr) and (curr_vix < high_thr)
# ... (Helpers unchanged) ...
# =========================
# STRATEGY HELPERS
# =========================
def get_nearest_strike(strikes, target):
    return min(strikes, key=lambda x: abs(x - target))

def get_nearest_expiry(expiry_list, ts):
    future_expiries = [e for e in expiry_list if e >= ts.date()]
    return min(future_expiries) if future_expiries else None

def compute_atr(df, period=ATR_PERIOD):
    high = df['High']
    low = df['Low']
    close = df['Close']
    prev_close = close.shift(1)
    tr = pd.concat([high - low, (high - prev_close).abs(), (low - prev_close).abs()], axis=1).max(axis=1)
    atr = tr.rolling(window=period, min_periods=1).mean()
    return atr

def price_breakout_confirmation(spot_data, ts, direction, lookback=CONFIRM_LOOKBACK, sma_period=21):
    window = spot_data.loc[:ts].iloc[-lookback:]
    if len(window) < lookback:
        return False
    last_close = window['close'].iloc[-1]
    mean_high = window['high'][:-1].mean()
    mean_low = window['low'][:-1].mean()
    sma21 = window['close'].rolling(window=sma_period, min_periods=1).mean().iloc[-1]
    if direction == 1:
        return last_close > mean_high and last_close > sma21
    elif direction == -1:
        return last_close < mean_low and last_close < sma21
    return False

def trend_filter(spot_data, ts, direction, sma_period=50):
    window = spot_data.loc[:ts].iloc[-sma_period:]
    if len(window) < sma_period:
        return False
    last_close = window['close'].iloc[-1]
    sma = window['close'].mean()
    if direction == 1:
        return last_close > sma
    elif direction == -1:
        return last_close < sma
    return False

def get_dynamic_tp(vix_val):
    if vix_val <= VIX_LOW + 1:
        return 0.15
    elif vix_val <= VIX_HIGH - 1:
        return 0.25
    else:
        return 0.35

def get_risk_per_trade_buy(vix_value):
    if vix_value <= BUY_VIX_LOW:
        return BUY_RISK_LOW
    elif vix_value <= BUY_VIX_MED:
        return BUY_RISK_MED
    else:
        return BUY_RISK_HIGH
    
def choose_best_liquid_strike(options_data, atm_strike, expiry, ts):
    today = ts.date()
    expiry = pd.to_datetime(expiry)
    filtered = options_data[
        (options_data['Expiry'] == expiry) &
        (options_data['TimeStamp'] <= ts) &
        (abs(options_data['Strike'] - atm_strike) <= 100)
    ].copy()
    if expiry.date() == today:
        return None, None, None, None
    if not filtered.empty:
        filtered['liq_score'] = filtered['OI'] + filtered['Volume']
        best_row = filtered.sort_values('liq_score', ascending=False).iloc[0]
        return int(best_row['Strike']), best_row['Type'], int(best_row['Volume']), int(best_row['OI'])
    return None, None, None, None

def calculate_realistic_costs(entry_price, lots, trade_type, option_volume, option_oi):
    brokerage = 20
    stt = 0.0 if trade_type == 'buy' else entry_price * lots * 50 * 0.001
    exchange_charges = max(20, entry_price * lots * 50 * 0.00053)
    if entry_price < 10 or option_volume < 100 or option_oi < 100:
        spread_impact = entry_price * 0.15
    elif entry_price < 50:
        spread_impact = entry_price * 0.08
    else:
        spread_impact = entry_price * 0.03
    base_slippage = 0.005
    size_penalty = min(0.002 * (lots / 10), 0.01)
    total_slippage = (base_slippage + size_penalty) * entry_price * lots * 50
    return brokerage + stt + exchange_charges + spread_impact + total_slippage

def check_liquidity_constraint(strike, option_type, lots, entry_price, atm_strike, option_volume, option_oi, current_time):
    strike_distance_from_atm = abs(strike - atm_strike)
    if strike_distance_from_atm > 300 and lots > 5:
        return False
    if entry_price < 5 and lots > 20:
        return False
    if current_time.hour == 9 and current_time.minute < 30:
        return random.random() < 0.7
    if option_oi < 50 or option_volume < 50:
        return False
    return True

def simulate_realistic_execution(order_size, option_volume, option_oi):
    if order_size > 50:
        fill_probability = max(0.6, 1 - (order_size - 50) * 0.01)
        if random.random() > fill_probability:
            return None
        else:
            fill_ratio = random.uniform(0.3, 0.8)
            return int(order_size * fill_ratio)
    if option_volume < 100 or option_oi < 100:
        fill_ratio = random.uniform(0.5, 1.0)
        return int(order_size * fill_ratio)
    return order_size

def calculate_realistic_position_size(entry_price, account_balance, risk_per_trade, option_volume, option_oi):
    base_lots = int((account_balance * risk_per_trade) / (entry_price * 50))
    if entry_price < 5:
        max_lots = 10
    elif entry_price < 20:
        max_lots = 25
    elif entry_price < 50:
        max_lots = 50
    else:
        max_lots = 100
    if option_volume < 100 or option_oi < 100:
        max_lots = min(max_lots, 5)
    final_lots = min(base_lots, max_lots)
    if random.random() < 0.1:
        final_lots = int(final_lots * random.uniform(0.5, 0.8))
    return max(1, final_lots)

def simulate_trading_problems():
    problems = {
        'data_feed_issues': 0.01,
        'broker_rejection': 0.03,
    }
    for problem, probability in problems.items():
        if random.random() < probability:
            return problem
    return None

def compute_iv_rank(option_df, window=252):
    option_df = option_df.copy()
    option_df['IVR'] = option_df['IV'].rolling(window).apply(
        lambda x: 100 * (x[-1] - x.min()) / (x.max() - x.min()) if (x.max() - x.min()) > 0 else 0,
        raw=True
    )
    return option_df

def valid_signal_for_side(side, ts, chunk_signals, vix_data, spot_data):
    if ts not in chunk_signals.index:
        return False
    signal = chunk_signals.loc[ts, 'Signal']
    if side == 'buy' and signal == 1:
        return True
    elif side == 'sell' and signal == -1:
        return True
    return False

def should_exit_trade(pos, ts, options_data):
    if 'expiry' in pos and pd.to_datetime(ts).date() >= pd.to_datetime(pos['expiry']).date():
        return True
    return False

def execute_exit(pos, ts, options_data):
    match = options_data[
        (options_data['Strike'] == pos['strike']) &
        (options_data['Type'] == pos['type']) &
        (options_data['Expiry'].dt.date == pd.to_datetime(pos['expiry']).date()) &
        (options_data['TimeStamp'] >= pos['entry_time']) &
        (options_data['TimeStamp'] <= ts)
    ].sort_values('TimeStamp')
    if match.empty:
        exit_price = pos['entry_price']
        exit_time = ts
    else:
        exit_price = float(match.iloc[-1]['Open'])
        exit_time = match.iloc[-1]['TimeStamp']
    lots = pos.get('lots', 1)
    direction = pos.get('action', 'Buy')
    if direction.lower() == 'buy':
        pnl = (exit_price - pos['entry_price']) * lot_size * lots
    else:
        pnl = (pos['entry_price'] - exit_price) * lot_size * lots
    brokerage = 20
    slippage = exit_price * lot_size * lots * 0.001
    trade_log = {
        'entry_date': pos['entry_date'],
        'entry_time': pos['entry_time'],
        'exit_date': exit_time,
        'action': direction,
        'type': pos['type'],
        'strike': pos['strike'],
        'entry_price': pos['entry_price'],
        'exit_price': exit_price,
        'lots': lots,
        'exit_pnl': pnl - brokerage - slippage,
        'spot': pos.get('spot', None),
        'expiry': pos['expiry'],
        'exit_reason': 'expiry',
        'brokerage': brokerage,
        'slippage': slippage
    }
    margin_released = pos.get('margin', pos['entry_price'] * lot_size * lots)
    return trade_log, pnl - brokerage - slippage, margin_released

def propose_trade(side, ts, chunk_options, spot_data, vix_data, available_cash):
    try:
        spot_row = spot_data.loc[ts]
        spot_price = spot_row['close']
        atm_strike = int(round(spot_price / 50) * 50)
        expiry_list = sorted(chunk_options['Expiry'].dt.date.unique())
        # Helper:
        def get_nearest_expiry(expiry_list, ts):
            future_expiries = [e for e in expiry_list if e >= ts.date()]
            return min(future_expiries) if future_expiries else None
        expiry = get_nearest_expiry(expiry_list, ts)
        if expiry is None:
            return None, 0, None, 0, 0
        trade_type = 'CE' if (side == 'buy' or side == 'sell') and spot_price >= atm_strike else 'PE'
        filtered = chunk_options[
            (chunk_options['Strike'] == atm_strike) &
            (chunk_options['Type'] == trade_type) &
            (chunk_options['Expiry'].dt.date == expiry) &
            (chunk_options['TimeStamp'] >= ts)
        ]
        if filtered.empty:
            return None, 0, None, 0, 0
        entry_row = filtered.iloc[0]
        entry_price = float(entry_row['Open'])
        lots = 1
        margin_required = max(15000, entry_price * lot_size * lots)
        if available_cash < margin_required:
            return None, 0, None, 0, 0
        action = 'Buy' if side == 'buy' else 'Sell'
        proposed_trade = {
            'entry_date': ts,
            'entry_time': entry_row['TimeStamp'],
            'action': action,
            'type': trade_type,
            'strike': atm_strike,
            'entry_price': entry_price,
            'lots': lots,
            'expiry': expiry,
            'margin': margin_required,
            'spot': spot_price
        }
        entry_log = dict(proposed_trade)
        brokerage = 20
        slippage = entry_price * lot_size * lots * 0.001
        return proposed_trade, margin_required, entry_log, brokerage, slippage
    except Exception:
        return None, 0, None, 0, 0


# =========================
# BUY-SIDE STRATEGY (REALISTIC)
# =========================
def execute_trades_15min_atr_trailing_vix_realistic(
    spot_data, signals, options_data, vix_data,
    partial_tp_pct=BUY_PARTIAL_TP_PCT, tp_pct_low=BUY_FINAL_TP_PCT_LOW, tp_pct_high=BUY_FINAL_TP_PCT_HIGH,
    brokerage_per_order=BUY_BROKERAGE_PER_ORDER, slippage_pct=BUY_SLIPPAGE_PCT
):
    portfolio_cash = initial_capital
    trade_log = []
    total_brokerage = 0
    total_slippage = 0
    expiry_list = sorted(options_data['Expiry'].dt.date.unique())
    signals_15min = signals.reindex(spot_data.index, method='ffill').dropna()
    vix_data = vix_data.reindex(spot_data.index, method='ffill').dropna()
    open_positions = []
    previous_entry_price = None
    max_profitable_lots = 0

    options_data['ATR'] = options_data.groupby(['Strike', 'Type', 'Expiry'])\
        .apply(lambda x: compute_atr(x, BUY_ATR_PERIOD)).reset_index(level=[0,1,2], drop=True)
    if 'IV' in options_data.columns:
        options_data = options_data.groupby(['Strike', 'Type', 'Expiry']).apply(compute_iv_rank).reset_index(drop=True)
    else:
        options_data['IVR'] = 0

    for ts, row in spot_data.iterrows():
        if ts.time() >= datetime.strptime(BUY_MARKET_CLOSE_TIME, '%H:%M:%S').time():
            for pos in open_positions[:]:
                exit_row = options_data[
                    (options_data['Strike'] == pos['strike']) &
                    (options_data['Type'] == pos['type']) &
                    (options_data['Expiry'].dt.date == pos['expiry']) &
                    (options_data['TimeStamp'] >= ts)
                ].sort_values('TimeStamp')
                if not exit_row.empty:
                    exit_price = float(exit_row.iloc[0]['Open'])
                    exit_time = exit_row.iloc[0]['TimeStamp']
                    costs = calculate_realistic_costs(exit_price, pos['lots'], 'buy', pos['option_volume'], pos['option_oi'])
                    pnl = (exit_price - pos['entry_price']) * lot_size * pos['lots'] - costs
                    if pnl < -BUY_MAX_LOSS:
                        pnl = -BUY_MAX_LOSS
                    portfolio_cash += pnl
                    trade_log.append({
                        'entry_date': pos['entry_date'],
                        'entry_time': pos['entry_time'],
                        'exit_date': exit_time,
                        'action': pos['action'],
                        'type': pos['type'],
                        'strike': pos['strike'],
                        'entry_price': pos['entry_price'],
                        'exit_price': exit_price,
                        'lots': pos['lots'],
                        'pnl': pnl,
                        'holding_days': 0,
                        'spot': pos['spot'],
                        'expiry': pos['expiry'],
                        'exit_reason': 'EOD Close'
                    })
                    if pnl > 0:
                        max_profitable_lots = max(max_profitable_lots, pos['lots'])
                    open_positions.remove(pos)
            continue

        if ts not in signals_15min.index or pd.isna(row['close']):
            continue
        signal = signals_15min.loc[ts, 'Signal']
        if signal == 0:
            continue
        spot_price = row['close']
        atm_strike = int(round(spot_price / 50) * 50)
        expiry = get_nearest_expiry(expiry_list, ts)
        if expiry is None:
            continue
        vix_val = vix_data.loc[ts, 'Close'] if ts in vix_data.index else None
        if vix_val is None or np.isnan(vix_val):
            continue
        if not adaptive_vix_trend_filter(vix_data['Close'], ts):
            continue
        vix_ma20 = vix_data['Close'].rolling(20*24*4, min_periods=1).mean().loc[ts]
        ivr = 0
        best_strike, best_type, option_volume, option_oi = choose_best_liquid_strike(options_data, atm_strike, expiry, ts)
        if best_strike is None:
            continue
        option_type = 'CE' if signal == 1 else 'PE'
        if vix_val >= vix_ma20 or ivr >= BUY_IVR_THRESHOLD:
            continue
        if vix_val <= BUY_VIX_LOW:
            final_tp_pct = tp_pct_low
        elif vix_val > BUY_VIX_MED:
            final_tp_pct = tp_pct_high
        else:
            final_tp_pct = tp_pct_low + (tp_pct_high - tp_pct_low) * ((vix_val - BUY_VIX_LOW)/(BUY_VIX_MED - BUY_VIX_LOW))
        risk_per_trade = get_risk_per_trade_buy(vix_val)
        if not any(pos['strike'] == best_strike and pos['expiry'] == expiry and pos['type'] == option_type for pos in open_positions):
            entry_row = options_data[
                (options_data['Strike'] == best_strike) &
                (options_data['Type'] == option_type) &
                (options_data['Expiry'].dt.date == expiry) &
                (options_data['TimeStamp'] >= ts)
            ].sort_values('TimeStamp')
            if entry_row.empty:
                continue
            entry_price = float(entry_row.iloc[0]['Open'])
            entry_time = entry_row.iloc[0]['TimeStamp']
            lots = calculate_realistic_position_size(entry_price, portfolio_cash, risk_per_trade, option_volume, option_oi)
            if previous_entry_price is not None and entry_price < 0.2 * previous_entry_price:
                if max_profitable_lots > 0:
                    lots = min(lots, max_profitable_lots)
            if not check_liquidity_constraint(best_strike, option_type, lots, entry_price, atm_strike, option_volume, option_oi, ts):
                continue
            lots_filled = simulate_realistic_execution(lots, option_volume, option_oi)
            if lots_filled is None or lots_filled < 1:
                continue
            if simulate_trading_problems() is not None:
                continue
            partial_tp_price = entry_price * (1 + partial_tp_pct)
            final_tp_price = entry_price * (1 + final_tp_pct)
            atr = float(entry_row.iloc[0]['ATR'])
            trailing_stop = entry_price - BUY_ATR_MULTIPLIER * atr
            open_positions.append({
                'entry_date': ts,
                'entry_time': entry_time,
                'action': 'Buy',
                'type': option_type,
                'strike': best_strike,
                'entry_price': entry_price,
                'lots': lots_filled,
                'spot': spot_price,
                'expiry': expiry,
                'signal': signal,
                'option_volume': option_volume,
                'option_oi': option_oi,
                'partial_tp_price': partial_tp_price,
                'final_tp_price': final_tp_price,
                'trailing_stop': trailing_stop,
                'max_loss_stop': entry_price - (BUY_MAX_LOSS / (lot_size * lots_filled)),
                'highest_price': entry_price,
                'risk_per_trade': risk_per_trade,
                'vix_at_entry': vix_val,
                'ivr_at_entry': ivr,
                'partial_booked': False,
                'partial_lots': lots_filled // 2,
                'final_lots': lots_filled - (lots_filled // 2)
            })
            previous_entry_price = entry_price

        for pos in open_positions[:]:
            exit_reason = None
            exit_price = None
            exit_time = None
            option_rows = options_data[
                (options_data['Strike'] == pos['strike']) &
                (options_data['Type'] == pos['type']) &
                (options_data['Expiry'].dt.date == pos['expiry']) &
                (options_data['TimeStamp'] >= pos['entry_time']) &
                (options_data['TimeStamp'] <= ts)
            ].sort_values('TimeStamp')
            for _, opt_row in option_rows.iterrows():
                price = float(opt_row['Open'])
                atr = float(opt_row['ATR'])
                if price > pos['highest_price']:
                    pos['highest_price'] = price
                trailing_stop = pos['highest_price'] - BUY_ATR_MULTIPLIER * atr
                pos['trailing_stop'] = max(pos['trailing_stop'], trailing_stop)
                if not pos['partial_booked'] and price >= pos['partial_tp_price']:
                    costs = calculate_realistic_costs(price, pos['partial_lots'], 'buy', pos['option_volume'], pos['option_oi'])
                    pnl = (price - pos['entry_price']) * lot_size * pos['partial_lots'] - costs
                    if pnl < -BUY_MAX_LOSS:
                        pnl = -BUY_MAX_LOSS
                    portfolio_cash += pnl
                    trade_log.append({
                        'entry_date': pos['entry_date'],
                        'entry_time': pos['entry_time'],
                        'exit_date': opt_row['TimeStamp'],
                        'action': pos['action'],
                        'type': pos['type'],
                        'strike': pos['strike'],
                        'entry_price': pos['entry_price'],
                        'exit_price': price,
                        'lots': pos['partial_lots'],
                        'pnl': pnl,
                        'holding_days': 0,
                        'spot': pos['spot'],
                        'expiry': pos['expiry'],
                        'exit_reason': 'Partial TP',
                        'risk_per_trade': pos['risk_per_trade'],
                        'vix_at_entry': pos['vix_at_entry'],
                        'ivr_at_entry': pos['ivr_at_entry']
                    })
                    if pnl > 0:
                        max_profitable_lots = max(max_profitable_lots, pos['partial_lots'])
                    pos['partial_booked'] = True
                    pos['final_lots'] = pos['final_lots']
                    continue
                if pos['partial_booked'] and price >= pos['final_tp_price']:
                    exit_price = pos['final_tp_price']
                    exit_time = opt_row['TimeStamp']
                    exit_reason = 'Final TP'
                    break
                elif pos['partial_booked'] and price <= pos['trailing_stop']:
                    exit_price = pos['trailing_stop']
                    exit_time = opt_row['TimeStamp']
                    exit_reason = 'Trailing Stop'
                    break
                elif pos['partial_booked'] and price <= pos['max_loss_stop']:
                    exit_price = pos['max_loss_stop']
                    exit_time = opt_row['TimeStamp']
                    exit_reason = 'Max Loss Stop'
                    break
            if pos['partial_booked'] and exit_reason is not None and exit_price is not None:
                costs = calculate_realistic_costs(exit_price, pos['final_lots'], 'buy', pos['option_volume'], pos['option_oi'])
                pnl = (exit_price - pos['entry_price']) * lot_size * pos['final_lots'] - costs
                if pnl < -BUY_MAX_LOSS:
                    pnl = -BUY_MAX_LOSS
                portfolio_cash += pnl
                trade_log.append({
                    'entry_date': pos['entry_date'],
                    'entry_time': pos['entry_time'],
                    'exit_date': exit_time,
                    'action': pos['action'],
                    'type': pos['type'],
                    'strike': pos['strike'],
                    'entry_price': pos['entry_price'],
                    'exit_price': exit_price,
                    'lots': pos['final_lots'],
                    'pnl': pnl,
                    'holding_days': 0,
                    'spot': pos['spot'],
                    'expiry': pos['expiry'],
                    'exit_reason': exit_reason,
                    'risk_per_trade': pos['risk_per_trade'],
                    'vix_at_entry': pos['vix_at_entry'],
                    'ivr_at_entry': pos['ivr_at_entry']
                })
                if pnl > 0:
                    max_profitable_lots = max(max_profitable_lots, pos['final_lots'])
                open_positions.remove(pos)
    return trade_log, total_brokerage, total_slippage

# =========================
# SELL-SIDE STRATEGY (ROBUST)
# =========================
def execute_sell_side(spot_data, signals, options_data, vix_data):
    portfolio_cash = initial_capital
    trade_log = []
    total_brokerage = 0
    total_slippage = 0
    available_strikes = np.sort(options_data['Strike'].unique())
    expiry_list = sorted(options_data['Expiry'].dt.date.unique())
    signals_15min = signals.reindex(spot_data.index, method='ffill').dropna()
    vix_data = vix_data.reindex(spot_data.index, method='ffill').dropna()
    open_spread = None
    prev_signal = 0
    atr_series = options_data.groupby(['Strike', 'Type', 'Expiry'], group_keys=False)[['High', 'Low', 'Close']].apply(
        lambda x: compute_atr(x, ATR_PERIOD)
    )
    options_data['ATR'] = atr_series.values 
    for ts, row in spot_data.iterrows():
        if ts not in signals_15min.index or pd.isna(row['close']):
            continue
        signal = signals_15min.loc[ts, 'Signal']
        spot_price = row['close']
        atm_strike = int(round(spot_price / 50) * 50)
        expiry = get_nearest_expiry(expiry_list, ts)
        vix_val = vix_data.loc[ts, 'Close'] if ts in vix_data.index else None
        if expiry is None or vix_val is None or np.isnan(vix_val):
            prev_signal = signal
            continue
        risk_per_trade = get_risk_per_trade_buy(vix_val)
        if not adaptive_vix_trend_filter(vix_data['Close'], ts):
            prev_signal = signal
            continue
        if not trend_filter(spot_data, ts, signal, sma_period=50):
            prev_signal = signal
            continue
        if open_spread is None and signal != 0 and signal != prev_signal and price_breakout_confirmation(spot_data, ts, signal):
            if signal == 1:
                short_strike = atm_strike
                long_strike = atm_strike - 100
                short_leg = options_data[
                    (options_data['Strike'] == short_strike) &
                    (options_data['Type'] == 'PE') &
                    (options_data['Expiry'].dt.date == expiry) &
                    (options_data['TimeStamp'] >= ts)
                ].sort_values('TimeStamp')
                long_leg = options_data[
                    (options_data['Strike'] == long_strike) &
                    (options_data['Type'] == 'PE') &
                    (options_data['Expiry'].dt.date == expiry) &
                    (options_data['TimeStamp'] >= ts)
                ].sort_values('TimeStamp')
            else:
                short_strike = atm_strike
                long_strike = atm_strike + 100
                short_leg = options_data[
                    (options_data['Strike'] == short_strike) &
                    (options_data['Type'] == 'CE') &
                    (options_data['Expiry'].dt.date == expiry) &
                    (options_data['TimeStamp'] >= ts)
                ].sort_values('TimeStamp')
                long_leg = options_data[
                    (options_data['Strike'] == long_strike) &
                    (options_data['Type'] == 'CE') &
                    (options_data['Expiry'].dt.date == expiry) &
                    (options_data['TimeStamp'] >= ts)
                ].sort_values('TimeStamp')
            if short_leg.empty or long_leg.empty:
                prev_signal = signal
                continue
            short_price = float(short_leg.iloc[0]['Open'])
            long_price = float(long_leg.iloc[0]['Open'])
            credit = (short_price - long_price) * lot_size
            margin_per_lot = abs(short_strike - long_strike) * lot_size - credit
            lots = int((portfolio_cash * risk_per_trade) // margin_per_lot)
            if lots < 1:
                prev_signal = signal
                continue
            tp_pct = get_dynamic_tp(vix_val)
            open_spread = {
                'entry_date': ts,
                'short_leg': {'strike': short_strike, 'type': short_leg.iloc[0]['Type'], 'expiry': expiry, 'price': short_price},
                'long_leg': {'strike': long_strike, 'type': long_leg.iloc[0]['Type'], 'expiry': expiry, 'price': long_price},
                'credit': credit,
                'lots': lots,
                'direction': 'bull_put' if signal == 1 else 'bear_call',
                'risk_per_trade': risk_per_trade,
                'vix_at_entry': vix_val,
                'tp_pct': tp_pct,
                'partial_booked': False
            }
            prev_signal = signal
            continue
        if open_spread is not None:
            short_leg, long_leg = open_spread['short_leg'], open_spread['long_leg']
            spread_rows_short = options_data[
                (options_data['Strike'] == short_leg['strike']) &
                (options_data['Type'] == short_leg['type']) &
                (options_data['Expiry'].dt.date == short_leg['expiry']) &
                (options_data['TimeStamp'] >= open_spread['entry_date']) &
                (options_data['TimeStamp'] <= ts)
            ].sort_values('TimeStamp')
            spread_rows_long = options_data[
                (options_data['Strike'] == long_leg['strike']) &
                (options_data['Type'] == long_leg['type']) &
                (options_data['Expiry'].dt.date == long_leg['expiry']) &
                (options_data['TimeStamp'] >= open_spread['entry_date']) &
                (options_data['TimeStamp'] <= ts)
            ].sort_values('TimeStamp')
            for i in range(min(len(spread_rows_short), len(spread_rows_long))):
                short_price = float(spread_rows_short.iloc[i]['Open'])
                long_price = float(spread_rows_long.iloc[i]['Open'])
                spread_value = (short_price - long_price) * lot_size
                pnl = (open_spread['credit'] - spread_value) * open_spread['lots']
                tp_level = open_spread['credit'] * open_spread['tp_pct']
                if not open_spread['partial_booked'] and pnl > tp_level:
                    booked_pnl = tp_level * PARTIAL_BOOK_PCT
                    portfolio_cash += booked_pnl
                    open_spread['credit'] -= booked_pnl
                    open_spread['lots'] = int(open_spread['lots'] * (1 - PARTIAL_BOOK_PCT))
                    open_spread['partial_booked'] = True
                    trade_log.append({**open_spread, 'exit_date': spread_rows_short.iloc[i]['TimeStamp'], 'exit_pnl': booked_pnl, 'exit_reason': 'Partial TP'})
                elif pnl > tp_level and open_spread['partial_booked']:
                    exit_time = spread_rows_short.iloc[i]['TimeStamp']
                    portfolio_cash += pnl
                    trade_log.append({**open_spread, 'exit_date': exit_time, 'exit_pnl': pnl, 'exit_reason': 'Full TP'})
                    open_spread = None
                    break
                elif pnl < -MAX_LOSS:
                    exit_time = spread_rows_short.iloc[i]['TimeStamp']
                    portfolio_cash += -MAX_LOSS
                    trade_log.append({**open_spread, 'exit_date': exit_time, 'exit_pnl': -MAX_LOSS, 'exit_reason': 'Max Loss'})
                    open_spread = None
                    break
                elif ts.date() >= open_spread['short_leg']['expiry']:
                    exit_time = spread_rows_short.iloc[i]['TimeStamp']
                    pnl = (open_spread['credit'] - spread_value) * open_spread['lots']
                    if pnl < -MAX_LOSS:
                        pnl = -MAX_LOSS
                    portfolio_cash += pnl
                    trade_log.append({**open_spread, 'exit_date': exit_time, 'exit_pnl': pnl, 'exit_reason': 'Expiry'})
                    open_spread = None
                    break
        prev_signal = signal
    return trade_log, total_brokerage, total_slippage

# ... (Reporting and main loop unchanged) ...
# =========================
# REPORTING AND VISUALIZATION
# =========================
def strategy_report_and_visualizations(trade_log_df, initial_capital, title_prefix=""):
    if trade_log_df.empty:
        print("No trades to report.")
        return
    portfolio_values = [initial_capital]
    portfolio_dates = [trade_log_df['entry_date'].iloc[0]]
    current_value = initial_capital
    for i, row in trade_log_df.iterrows():
        current_value += row['exit_pnl']
        portfolio_values.append(current_value)
        portfolio_dates.append(row['exit_date'])
    returns = pd.Series(portfolio_values).pct_change(fill_method=None).dropna()



    rf = 0.01  # risk-free rate (annual)
    sharpe = ((returns.mean() - rf/252) / returns.std()) * np.sqrt(252) if returns.std() > 0 else 0
    downside_std = returns[returns < 0].std()
    sortino = ((returns.mean() - rf/252) / downside_std) * np.sqrt(252) if downside_std > 0 else 0
    cumulative = pd.Series(portfolio_values)
    rolling_max = cumulative.cummax()
    drawdown = (cumulative - rolling_max) / rolling_max
    max_drawdown = abs(drawdown.min()) * initial_capital
    years = (portfolio_dates[-1] - portfolio_dates[0]).days / 365.25 if isinstance(portfolio_dates[0], pd.Timestamp) else len(portfolio_values) / 252
    cagr = (portfolio_values[-1]/portfolio_values[0])**(1/years) - 1 if years > 0 else 0
    calmar = cagr / abs(drawdown.min()) if drawdown.min() != 0 else 0

    print(f"""
{title_prefix} Strategy Performance Report:
-----------------------------
Initial Capital:    â‚¹{initial_capital:,.2f}
Final Portfolio:    â‚¹{portfolio_values[-1]:,.2f}
Total Return:       {(portfolio_values[-1]/initial_capital-1)*100:.2f}%
Max Drawdown:       â‚¹{max_drawdown:,.2f}
Sharpe Ratio:       {sharpe:.2f}
Sortino Ratio:      {sortino:.2f}
Calmar Ratio:       {calmar:.2f}

Trade Statistics:
-----------------
Total Trades:       {len(trade_log_df)}
Winning Trades:     {(trade_log_df['exit_pnl'] > 0).sum()}
Losing Trades:      {(trade_log_df['exit_pnl'] <= 0).sum()}
Avg PnL:            â‚¹{trade_log_df['exit_pnl'].mean():,.2f}
Best Trade:         â‚¹{trade_log_df['exit_pnl'].max():,.2f}
Worst Trade:        â‚¹{trade_log_df['exit_pnl'].min():,.2f}
Trade Period:       {trade_log_df['entry_date'].min().date()} to {trade_log_df['exit_date'].max().date()}
""")

    sns.set_style("whitegrid")
    plt.figure(figsize=(14, 7))
    plt.plot(portfolio_dates, portfolio_values, label='Portfolio Value', linewidth=2)
    plt.title(f'{title_prefix} Portfolio Value Over Time', fontsize=16)
    plt.ylabel('Value (â‚¹)', fontsize=14)
    plt.xlabel('Date', fontsize=14)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(14, 5))
    plt.plot(cumulative, label='Portfolio Value')
    plt.plot(rolling_max, '--', label='Rolling Max')
    plt.fill_between(cumulative.index, cumulative, rolling_max, color='red', alpha=0.3, label='Drawdown')
    plt.title(f"{title_prefix} Equity Curve with Drawdown")
    plt.legend()
    plt.show()

    plt.figure(figsize=(7, 4))
    sns.histplot(trade_log_df['exit_pnl'], bins=20, kde=True)
    plt.title(f"{title_prefix} Trade PnL Distribution")
    plt.xlabel("PnL (â‚¹)")
    plt.ylabel("Count")
    plt.show()
    
    # --- Heatmap: Buy/Sell by week and direction ---
    trade_log_df['exit_week'] = pd.to_datetime(trade_log_df['exit_date']).dt.isocalendar().week
    if 'direction' in trade_log_df.columns:
        trade_log_df['side'] = trade_log_df['direction'].map({
            'bull_put': 'Sell',
            'bear_call': 'Sell',
            'bull_call': 'Buy',
            'bear_put': 'Buy',
            'buy_call': 'Buy',
            'buy_put': 'Buy'
        }).fillna('Other')
    elif 'action' in trade_log_df.columns:
        trade_log_df['side'] = trade_log_df['action'].str.capitalize()
    else:
        trade_log_df['side'] = 'Unknown'
    heatmap_data = trade_log_df.pivot_table(index='exit_week', columns='side', values='exit_pnl', aggfunc='sum')
    plt.figure(figsize=(8, 5))
    sns.heatmap(heatmap_data, annot=True, fmt=".0f", cmap='RdYlGn')
    plt.title(f"{title_prefix} PnL Heatmap by Week and Side")
    plt.xlabel("Side")
    plt.ylabel("ISO Week")
    plt.show()

    # --- Monte Carlo Simulation ---
    n_sim = 1000
    sim_length = len(trade_log_df)
    sim_results = []
    for _ in range(n_sim):
        sim_pnls = np.random.choice(trade_log_df['exit_pnl'], size=sim_length, replace=True)
        sim_curve = np.cumsum(np.insert(sim_pnls, 0, initial_capital))
        sim_results.append(sim_curve[-1])
    plt.figure(figsize=(10, 5))
    plt.hist(sim_results, bins=30, color='skyblue')
    plt.axvline(np.mean(sim_results), color='red', linestyle='dashed', linewidth=2, label='Mean Final Value')
    plt.title(f"{title_prefix} Monte Carlo: Distribution of Final Portfolio Value")
    plt.xlabel("Final Portfolio Value (â‚¹)")
    plt.ylabel("Frequency")
    plt.legend()
    plt.show()
    print(f"Monte Carlo: Mean Final Value = â‚¹{np.mean(sim_results):,.2f}, 5th Percentile = â‚¹{np.percentile(sim_results, 5):,.2f}, 95th Percentile = â‚¹{np.percentile(sim_results, 95):,.2f}")
# Drop NaT from plotting arrays
clean_pairs = [(d, v) for d, v in zip(portfolio_dates, portfolio_values) if pd.notna(d) and not pd.isna(v)]
if not clean_pairs:
    print("No valid points for plot.")
    return
portfolio_dates, portfolio_values = zip(*clean_pairs)
portfolio_dates = list(portfolio_dates)
portfolio_values = list(portfolio_values)

# Convert all dates to timezone-naive for matplotlib compatibility
portfolio_dates = [pd.Timestamp(d).tz_localize(None) if pd.Timestamp(d).tzinfo is not None else pd.Timestamp(d) for d in portfolio_dates]

def combined_cumulative_report_and_visualizations(sell_log_df, buy_log_df, initial_capital):
    combined_df = pd.concat([sell_log_df, buy_log_df], ignore_index=True)
    if combined_df.empty:
        print("No trades to report in combined logs.")
        return
    combined_df = combined_df.sort_values('exit_date')
    portfolio_values = [initial_capital]
    portfolio_dates = [combined_df['entry_date'].min()]
    current_value = initial_capital
    for i, row in combined_df.iterrows():
        current_value += row['exit_pnl']
        portfolio_values.append(current_value)
        portfolio_dates.append(row['exit_date'])
    returns = pd.Series(portfolio_values).pct_change(fill_method=None).dropna()

    rf = 0.01
    sharpe = ((returns.mean() - rf/252) / returns.std()) * np.sqrt(252) if returns.std() > 0 else 0
    downside_std = returns[returns < 0].std()
    sortino = ((returns.mean() - rf/252) / downside_std) * np.sqrt(252) if downside_std > 0 else 0
    cumulative = pd.Series(portfolio_values)
    rolling_max = cumulative.cummax()
    drawdown = (cumulative - rolling_max) / rolling_max
    max_drawdown = abs(drawdown.min()) * initial_capital
    years = (portfolio_dates[-1] - portfolio_dates[0]).days / 365.25 if isinstance(portfolio_dates[0], pd.Timestamp) else len(portfolio_values) / 252
    cagr = (portfolio_values[-1]/portfolio_values[0])**(1/years) - 1 if years > 0 else 0
    calmar = cagr / abs(drawdown.min()) if drawdown.min() != 0 else 0
    print(f"""
Combined Strategy Performance Report:
-----------------------------
Initial Capital:    {initial_capital:,.2f}
Final Portfolio:    {portfolio_values[-1]:,.2f}
Total Return:       {(portfolio_values[-1]/initial_capital-1)*100:.2f}%
Max Drawdown:       {max_drawdown:,.2f}
Sharpe Ratio:       {sharpe:.2f}
Sortino Ratio:      {sortino:.2f}
Calmar Ratio:       {calmar:.2f}
Total Trades:       {len(combined_df)}
Winning Trades:     {(combined_df['exit_pnl'] > 0).sum()}
Losing Trades:      {(combined_df['exit_pnl'] <= 0).sum()}
Avg PnL:            {combined_df['exit_pnl'].mean():,.2f}
Best Trade:         {combined_df['exit_pnl'].max():,.2f}
Worst Trade:        {combined_df['exit_pnl'].min():,.2f}
Trade Period:       {combined_df['entry_date'].min().date()} to {combined_df['exit_date'].max().date()}
""")
    sns.set_style("whitegrid")
    plt.figure(figsize=(14, 7))
    plt.plot(portfolio_dates, portfolio_values, label='Portfolio Value', linewidth=2)
    plt.title('Combined Portfolio Value Over Time', fontsize=16)
    plt.ylabel('Value', fontsize=14)
    plt.xlabel('Date', fontsize=14)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()
    plt.figure(figsize=(14, 5))
    plt.plot(cumulative, label='Portfolio Value')
    plt.plot(rolling_max, '--', label='Rolling Max')
    plt.fill_between(cumulative.index, cumulative, rolling_max, color='red', alpha=0.3, label='Drawdown')
    plt.title('Combined Equity Curve with Drawdown')
    plt.legend()
    plt.show()
    plt.figure(figsize=(7, 4))
    sns.histplot(combined_df['exit_pnl'], bins=20, kde=True)
    plt.title('Combined Trade PnL Distribution')
    plt.xlabel('PnL')
    plt.ylabel('Count')
    plt.show()
    # --- Heatmap: Buy/Sell by week and side ---
    
    combined_df['exit_week'] = pd.to_datetime(combined_df['exit_date']).dt.isocalendar().week
    if 'direction' in combined_df.columns:
        combined_df['side'] = combined_df['direction'].map({
            'bull_put': 'Sell',
            'bear_call': 'Sell',
            'bull_call': 'Buy',
            'bear_put': 'Buy',
            'buy_call': 'Buy',
            'buy_put': 'Buy'
        }).fillna('Other')
    elif 'action' in combined_df.columns:
        combined_df['side'] = combined_df['action'].str.capitalize()
    else:
        combined_df['side'] = 'Unknown'
    heatmap_data = combined_df.pivot_table(index='exit_week', columns='side', values='exit_pnl', aggfunc='sum')
    plt.figure(figsize=(8, 5))
    sns.heatmap(heatmap_data, annot=True, fmt=".0f", cmap='RdYlGn')
    plt.title("Combined PnL Heatmap by Week and Side")
    plt.xlabel("Side")
    plt.ylabel("ISO Week")
    plt.show()
    n_sim = 1000
    sim_length = len(combined_df)
    sim_results = []
    for _ in range(n_sim):
        sim_pnls = np.random.choice(combined_df['exit_pnl'], size=sim_length, replace=True)
        sim_curve = np.cumsum(np.insert(sim_pnls, 0, initial_capital))
        sim_results.append(sim_curve[-1])
    plt.figure(figsize=(10, 5))
    plt.hist(sim_results, bins=30, color='skyblue')
    plt.axvline(np.mean(sim_results), color='red', linestyle='dashed', linewidth=2, label='Mean Final Value')
    plt.title('Combined Monte Carlo: Distribution of Final Portfolio Value')
    plt.xlabel('Final Portfolio Value')
    plt.ylabel('Frequency')
    plt.legend()
    plt.show()
    print(f"Monte Carlo: Mean Final Value = {np.mean(sim_results):,.2f}, 5th Percentile = {np.percentile(sim_results, 5):,.2f}, 95th Percentile = {np.percentile(sim_results, 95):,.2f}")
# Drop NaT from plotting arrays
clean_pairs = [(d, v) for d, v in zip(portfolio_dates, portfolio_values) if pd.notna(d) and not pd.isna(v)]
if not clean_pairs:
    print("No valid points for plot.")
    return
portfolio_dates, portfolio_values = zip(*clean_pairs)
portfolio_dates = list(portfolio_dates)
portfolio_values = list(portfolio_values)

# Convert all dates to timezone-naive for matplotlib compatibility
portfolio_dates = [pd.Timestamp(d).tz_localize(None) if pd.Timestamp(d).tzinfo is not None else pd.Timestamp(d) for d in portfolio_dates]

def combined_cumulative_heatmap(sell_log_df, buy_log_df, title_prefix="Cumulative"):
    combined_df = pd.concat([sell_log_df, buy_log_df], ignore_index=True)
    if combined_df.empty:
        print("No trades to report in combined logs.")
        return
    combined_df['exit_week'] = pd.to_datetime(combined_df['exit_date']).dt.isocalendar().week
    if 'direction' in combined_df.columns:
        combined_df['side'] = combined_df['direction'].map({
            'bull_put': 'Sell',
            'bear_call': 'Sell',
            'bull_call': 'Buy',
            'bear_put': 'Buy',
            'buy_call': 'Buy',
            'buy_put': 'Buy'
        }).fillna('Other')
    elif 'action' in combined_df.columns:
        combined_df['side'] = combined_df['action'].str.capitalize()
    else:
        combined_df['side'] = 'Unknown'
    heatmap_data = combined_df.pivot_table(index='exit_week', columns='side', values='exit_pnl', aggfunc='sum')
    plt.figure(figsize=(10, 6))
    sns.heatmap(heatmap_data, annot=True, fmt=".0f", cmap='RdYlGn')
    plt.title(f"{title_prefix} PnL Heatmap by Week and Side")
    plt.xlabel("Side")
    plt.ylabel("ISO Week")
    plt.show()

# =========================
# MAIN CHUNKED BACKTEST LOOP: UNIFIED CASH MANAGEMENT
# =========================

# --- Do not change: data loading and chunk setup ---
options_sample = load_options_sample()
start_date = options_sample['TimeStamp'].min()
final_expiry = options_sample['Expiry'].max()
start_date = pd.Timestamp(start_date).tz_localize('Asia/Kolkata') if start_date.tzinfo is None else start_date
final_end_date = pd.Timestamp(final_expiry + pd.Timedelta(days=1), tz='Asia/Kolkata')
chunk_length = pd.DateOffset(months=2)
chunk_idx = 1
current_start = start_date

portfolio = UnifiedPortfolio(initial_capital)
options_sample = load_options_sample()
start_date = options_sample['TimeStamp'].min()
final_expiry = options_sample['Expiry'].max()
start_date = pd.Timestamp(start_date).tz_localize('Asia/Kolkata') if start_date.tzinfo is None else start_date
final_end_date = pd.Timestamp(final_expiry + pd.Timedelta(days=1), tz='Asia/Kolkata')
chunk_length = pd.DateOffset(months=2)
chunk_idx = 1
current_start = start_date

while current_start < final_end_date:
    current_end = min(current_start + chunk_length - pd.Timedelta(minutes=1), final_end_date)
    print(f"\n=== Processing chunk {chunk_idx}: {current_start} to {current_end} ===")
    spot_data = load_spot(current_start, current_end)
    vix_data = load_vix(current_start, current_end)
    signals = load_signals_15min()
    chunk_signals = signals.loc[(signals.index >= current_start) & (signals.index <= current_end)]
    chunk_options = options_sample[
        (options_sample['TimeStamp'] >= current_start) &
        (options_sample['TimeStamp'] <= current_end)
    ]
    if spot_data.empty or vix_data.empty or chunk_signals.empty or chunk_options.empty:
        print(f"Chunk {chunk_idx} skipped due to missing data.")
        current_start = current_end + pd.Timedelta(minutes=1)
        chunk_idx += 1
        continue
    for ts in spot_data.index:
        # --- Exits ---
        for pos in portfolio.open_positions[:]:
            if should_exit_trade(pos, ts, chunk_options):
                exit_trade_log, pnl, margin_return = execute_exit(pos, ts, chunk_options)
                portfolio.release(margin_return)
                portfolio.cash += pnl
                portfolio.add_trade(exit_trade_log)
                portfolio.open_positions.remove(pos)
                portfolio.add_costs(exit_trade_log.get("brokerage", 0), exit_trade_log.get("slippage", 0))
        # --- Entries ---
        for side in ["buy", "sell"]:
            if valid_signal_for_side(side, ts, chunk_signals, vix_data, spot_data):
                proposed_trade, required_margin, entry_log, brokerage, slippage = propose_trade(
                    side, ts, chunk_options, spot_data, vix_data, portfolio.cash
                )
                if proposed_trade and portfolio.can_execute(required_margin):
                    portfolio.allocate(required_margin)
                    portfolio.open_positions.append(proposed_trade)
                    portfolio.add_trade(entry_log)
                    portfolio.add_costs(brokerage, slippage)
    # --- Per-Chunk Performance ---
    trade_df = pd.DataFrame(portfolio.trade_log)
    strategy_report_and_visualizations(trade_df, initial_capital, title_prefix=f"Unified Portfolio Chunk {chunk_idx}")
    current_start = current_end + pd.Timedelta(minutes=1)
    chunk_idx += 1


# =========================
# FINAL REPORTING (UPDATED)
# =========================
final_portfolio_value = portfolio.portfolio_value()
print(f"\n========= FINAL PORTFOLIO RESULT =========")
print(f"Ending Capital:           ₹{final_portfolio_value:,.2f}")
print(f"Total Trades:             {len(portfolio.trade_log)}")
print(f"Total Brokerage Paid:     ₹{portfolio.brokerage:,.2f}")
print(f"Total Slippage Paid:      ₹{portfolio.slippage:,.2f}")
print(f"Net Return:               {(final_portfolio_value/initial_capital-1)*100:.2f}%")

trade_df = pd.DataFrame(portfolio.trade_log)
# --- Clean for plot ---
trade_df = trade_df.dropna(subset=['exit_pnl', 'exit_date', 'entry_date']).sort_values('exit_date')
portfolio_values = [initial_capital]
portfolio_dates = [trade_df['entry_date'].iloc[0]]
current_value = initial_capital
for i, row in trade_df.iterrows():
    if pd.isna(row['exit_pnl']) or pd.isna(row['exit_date']):
        continue
    current_value += row['exit_pnl']
    portfolio_values.append(current_value)
    portfolio_dates.append(row['exit_date'])
# Drop NaT
pairs = [(d, v) for d, v in zip(portfolio_dates, portfolio_values) if pd.notna(d) and not pd.isna(v)]
if pairs:
    portfolio_dates, portfolio_values = zip(*pairs)
    portfolio_dates = [pd.Timestamp(d).tz_localize(None) if pd.Timestamp(d).tzinfo is not None else pd.Timestamp(d) for d in portfolio_dates]
    plt.figure(figsize=(14, 7))
    plt.plot(portfolio_dates, portfolio_values, label='Unified Portfolio Value', linewidth=2)
    plt.title("Unified Portfolio Value Over Time")
    plt.xlabel("Date")
    plt.ylabel("Portfolio Value (₹)")
    plt.legend()
    plt.show()
else:
    print('No valid data for equity curve plot.')


NameError: name 'portfolio_dates' is not defined